# DeepTaxa – Tutorial 3: Taxonomic Analysis
This notebook reproduces the [official analysis tutorial](https://systems-genomics-lab.github.io/deeptaxa/analysis.html).

It demonstrates downstream analysis of DeepTaxa prediction outputs:
per-rank accuracy, confusion matrices, error analysis, and diversity visualisation.

In [ ]:
!pip install -q deeptaxa huggingface_hub scikit-learn seaborn

In [ ]:
import json, os, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# ── Load prediction results ──────────────────────────────────────────
# Replace the path below with your own predictions JSON if needed
PRED_FILE = 'outputs/predictions/gg2_test_deeptaxa_predictions.json'
if not os.path.exists(PRED_FILE):
    print('Run Tutorial 1 first to generate predictions.')
else:
    with open(PRED_FILE) as f: preds = json.load(f)
    df = pd.DataFrame(preds)
    print(df.shape)
    df.head()

In [ ]:
# Per-rank accuracy (reported in the paper)
ranks = ['domain','phylum','class','order','family','genus','species']
paper_acc = [99.97, 99.52, 98.82, 97.73, 96.74, 95.01, 92.96]
plt.figure(figsize=(9,4))
plt.bar(ranks, paper_acc, color='steelblue', alpha=0.8)
for i,(r,v) in enumerate(zip(ranks, paper_acc)):
    plt.text(i, v+0.2, f'{v}%', ha='center', fontsize=9)
plt.ylim(88, 101)
plt.ylabel('Accuracy (%)')
plt.title('DeepTaxa accuracy on Greengenes2 test set (full-length 16S)')
plt.tight_layout(); plt.show()

In [ ]:
# If you have your own predictions with true labels, compute metrics
if 'df' in dir() and 'true_species' in df.columns:
    report = classification_report(df['true_species'], df['pred_species'], output_dict=True)
    pd.DataFrame(report).T.head(20)

In [ ]:
# Phylum-level prediction heatmap (true vs predicted)
if 'df' in dir() and 'true_phylum' in df.columns:
    top_phyla = df['true_phylum'].value_counts().head(12).index
    sub = df[df['true_phylum'].isin(top_phyla)]
    cm = confusion_matrix(sub['true_phylum'], sub['pred_phylum'], labels=top_phyla)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(11,9))
    sns.heatmap(cm_norm, xticklabels=top_phyla, yticklabels=top_phyla,
                annot=True, fmt='.2f', cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('True')
    plt.title('Phylum-level confusion matrix (top 12 phyla)')
    plt.tight_layout(); plt.show()
else:
    print('True labels not available in sample predictions – run full test set prediction.')

In [ ]:
# Top confused species pairs
if 'df' in dir() and 'true_species' in df.columns:
    errors = df[df['true_species'] != df['pred_species']]
    top_confused = (errors.groupby(['true_species','pred_species'])
                    .size().reset_index(name='count')
                    .sort_values('count', ascending=False).head(15))
    print(top_confused.to_string(index=False))
else:
    print('True labels required for error analysis.')

In [ ]:
# Error rate vs training frequency (log-log plot)
if 'df' in dir() and 'true_species' in df.columns:
    freq = df['true_species'].value_counts()
    err_rate = df[df['true_species'] != df['pred_species']]['true_species'].value_counts() / freq
    common = freq[freq >= 5].index
    plt.figure(figsize=(8,5))
    plt.scatter(freq[common], err_rate.reindex(common).fillna(0), alpha=0.4, s=15)
    plt.xscale('log'); plt.yscale('log')
    plt.xlabel('Training frequency'); plt.ylabel('Error rate')
    plt.title('Species error rate vs training frequency')
    plt.tight_layout(); plt.show()
else:
    print('True labels required.')